In [6]:
import numpy as np
import scipy.stats as stats
import src.noise as noise
import src.release as dprelease

from src.analysis import NoisyContingencyTable

Experimental parameters, no need to understand them at the outset.

In [3]:
# Standard deviation of Gaussian DP noise
NOISE_SCALE = 12.0

# Monte Carlo draws for CI estimation
MC_DRAWS = 1000

# Times to repeat the experiment
N_TRIALS = 500

Here's a contingency table.

In [4]:
true_tbl = np.array([
    [65, 109],
    [243, 1348]
])

It's straightforward to have `scipy` calculate the 95% confidence interval of its odds ratio.

In [5]:
scipy_ci = stats.contingency.odds_ratio(true_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.363633, 4.629785)


The `dpvalue` library can do this too. Although it uses a _credible interval_ because it's drawing from a posterior. It implicitly converts `true_tbl` into a table with elements of type `NoisyFloat` (the noise is just zero).

In [7]:
nct = NoisyContingencyTable(true_tbl)
dpvalue_lo, dpvalue_hi = nct.odds_ratio().credible_interval(n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (3.308038, 3.308038)


In [6]:
import src.core as core
print(f"Binomial sample count: {core._binomial_sample_count}")
print(f"Normal sample count: {core._normal_sample_count}")

Binomial sample count: 2000
Normal sample count: 0


More or less consistent, I guess.

Let's add some noise and pretend the contingency table is a differentially private release.

In [4]:
noise_factory = noise.gaussian(loc=0.0, scale=NOISE_SCALE)
noisy_tbl = dprelease.noisy_float_array(true_tbl, noise_factory)
noisy_tbl

array([[~47.85264791485261, ~91.78132503990231],
       [~242.953027069234, ~1381.8539773489813]], dtype=object)

What would `scipy` do with this? Since `scipy` isn't DP-noise-aware, it can only take the observed values as truth and account for sampling uncertainty as it did before.

But oops, we can't use `scipy` on this directly, even though `NoisyFloat` is castable to `float`. The elements are counts, and have to be type `int`. Let's extract the noisy observations.

In [7]:
noisy_int_tbl = [
    [int(noisy_tbl[0, 0]._obs), int(noisy_tbl[0, 1]._obs)],
    [int(noisy_tbl[1, 0]._obs), int(noisy_tbl[1, 1]._obs)]]

noisy_int_tbl

[[54, 119], [253, 1344]]

And here's the `scipy` confidence interval again.

In [8]:
scipy_ci = stats.contingency.odds_ratio(noisy_int_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (1.701251, 3.415724)


But the `dpvalue` library can use `noisy_tbl` directly. It accounts for both uncertainty from DP noise _and_ uncertainty from sampling.

In [5]:
dpvalue_lo, dpvalue_hi = odds_ratio(noisy_tbl).credible_interval(p=0.95, n=1000)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (1.201283, 5.505580)


In [6]:
import src.core as core
print(f"Binomial sample count: {core._binomial_sample_count}")
print(f"Normal sample count: {core._normal_sample_count}")

Binomial sample count: 2000
Normal sample count: 8


And because of this, it knows the 95% odds ratio confidence interval for the DP release is actually a little bit wider.

How can we confirm if this interval is correct? Well, we know `true_tbl` odds ratio...

In [10]:
n0 = int(true_tbl[0, 0] + true_tbl[0, 1])
n1 = int(true_tbl[1, 0] + true_tbl[1, 1])
p0 = true_tbl[0, 0] / n0
p1 = true_tbl[1, 0] / n1

true_or = (p0 / (1.0 - p0)) / (p1 / (1.0 - p1))

We'll recreate `noisy_tbl` many times with fresh differential privacy noise draws _and_ fresh counts based on `n0`, `p0`, `n1`, and `p1`. We'll have `dpnoise` and `scipy` calculate 95% confidence intervals for each `noisy_tbl` we create. We'll count the number of times the confidence interval cover the true odds ratio.

In [11]:
dp_covers = 0
scipy_covers = 0

By definition, a 95% credible interval should contain the true odds ratio about 95% of the time under repeated simulation.

This is a posterior calibration check, not a frequentist confidence-interval guarantee.

In [12]:
dp_contains_true = 0

for _ in range(N_TRIALS):
    # Simulate an ordinary (non-private) contingency table.
    grp0_yes = np.random.binomial(n0, p0)
    grp1_yes = np.random.binomial(n1, p1)
    true_tbl = np.array([
        [grp0_yes, n0 - grp0_yes],
        [grp1_yes, n1 - grp1_yes],
    ])

    # Differentially private release of true_tbl
    noisy_sim_tbl = dprelease.noisy_float_array(true_tbl, noise_factory)

    # Noise-aware posterior credible interval (dpvalue)
    lo_dp, hi_dp = odds_ratio(noisy_sim_tbl).credible_interval(p=0.95, n=MC_DRAWS)
    dp_contains_true += (lo_dp <= true_or <= hi_dp)

KeyboardInterrupt: 

In [ ]:
print(f"Target odds ratio: {true_or:.4f}")
print(f"Replications: {N_TRIALS}, DP noise scale: {NOISE_SCALE}, MC draws/rep: {MC_DRAWS}")
print(f"dpvalue credible interval coverage: {dp_contains_true/N_TRIALS:.3f}")

Target odds ratio: 3.3080
Replications: 500, DP noise scale: 6.0, MC draws/rep: 2000
dpvalue (noise-aware): 0.958
scipy on noisy counts (noise-unaware baseline): 0.912


So not exactly 0.95 for `dpvalue` but close enough to attribute the remainder to chance? The `scipy` confidence intervals are too narrow, they only contain the true value ~91% of the time. Probably more noise would amplify the difference.